# About
This point of this notebook is to properly format the namings in the catalogue & inventory.
(theres fuzzy differences in the strings)

Borrowings & inventory share the same titles, so: catalogue will attain these titles too.

But since the author names of catalogue are more reliable (fetched via an API) we will use them in the inventory.

In short:

Catalogue will get the titles of inventory
Inventory will get the authors of catalogue

In [1]:
import pandas as pd
from fuzzywuzzy import fuzz, process
import logging


C:\Users\fafao\AppData\Roaming\Python\Python313\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_data(inventory_file, catalogue_file):
    """Load inventory and catalogue CSV files"""
    try:
        inventory = pd.read_csv(inventory_file)
        catalogue = pd.read_csv(catalogue_file)
        logger.info(f"Inventory loaded: {len(inventory)} records")
        logger.info(f"Catalogue loaded: {len(catalogue)} records")
        return inventory, catalogue
    except Exception as e:
        logger.error(f"Error loading files: {e}")
        raise

def find_best_match(query, choices, threshold=80):
    """Find the best fuzzy match for a query among choices"""
    if not choices or pd.isna(query):
        return None, 0
    
    best_match, score = process.extractOne(str(query), choices, scorer=fuzz.token_sort_ratio)
    return best_match if score >= threshold else None, score

def integrate_data(inventory, catalogue, title_threshold=85, author_threshold=80):
    """
    Integrate inventory and catalogue data:
    - Catalogue gets titles from inventory
    - Inventory gets authors from catalogue
    """
    # Create copies to avoid modifying originals
    inventory_integrated = inventory.copy()
    catalogue_integrated = catalogue.copy()
    
    # Create dictionaries for faster lookups
    # Assuming the files have 'title' and 'author' columns
    # Adjust column names as needed
    
    # Prepare inventory lookup by title
    inventory_titles = inventory['Title'].dropna().unique().tolist()
    inventory_by_title = inventory.set_index('Title').to_dict('index')
    
    # Prepare catalogue lookup by author
    catalogue_authors = catalogue['Author'].dropna().unique().tolist()
    catalogue_by_author = catalogue.set_index('Author').to_dict('index')
    
    # Track statistics
    stats = {
        'title_matches': 0,
        'author_matches': 0,
        'title_misses': 0,
        'author_misses': 0
    }
    
    # 1. Update catalogue titles from inventory
    logger.info("Updating catalogue titles from inventory...")
    for idx, cat_row in catalogue_integrated.iterrows():
        cat_title = cat_row['Title']
        
        if pd.isna(cat_title):
            continue
            
        # Find best matching title in inventory
        best_match, score = find_best_match(cat_title, inventory_titles, title_threshold)
        
        if best_match and best_match in inventory_by_title:
            # Update catalogue title with inventory title
            inventory_title = inventory_by_title[best_match]['Title']
            catalogue_integrated.at[idx, 'Title'] = inventory_title
            stats['title_matches'] += 1
            logger.debug(f"Catalogue title '{cat_title}' -> '{inventory_title}' (score: {score})")
        else:
            stats['title_misses'] += 1
            logger.debug(f"No good match found for catalogue title: '{cat_title}' (best score: {score})")
    
    # 2. Update inventory authors from catalogue
    logger.info("Updating inventory authors from catalogue...")
    for idx, inv_row in inventory_integrated.iterrows():
        inv_author = inv_row['Author']
        
        if pd.isna(inv_author):
            continue
            
        # Find best matching author in catalogue
        best_match, score = find_best_match(inv_author, catalogue_authors, author_threshold)
        
        if best_match and best_match in catalogue_by_author:
            # Update inventory author with catalogue author
            catalogue_author = catalogue_by_author[best_match]['Author']
            inventory_integrated.at[idx, 'Author'] = catalogue_author
            stats['author_matches'] += 1
            logger.debug(f"Inventory author '{inv_author}' -> '{catalogue_author}' (score: {score})")
        else:
            stats['author_misses'] += 1
            logger.debug(f"No good match found for inventory author: '{inv_author}' (best score: {score})")
    
    logger.info(f"Title matches: {stats['title_matches']}/{len(catalogue)}")
    logger.info(f"Title misses: {stats['title_misses']}")
    logger.info(f"Author matches: {stats['author_matches']}/{len(inventory)}")
    logger.info(f"Author misses: {stats['author_misses']}")
    
    return inventory_integrated, catalogue_integrated, stats

def save_results(inventory_df, catalogue_df, inventory_output, catalogue_output):
    """Save the integrated data to CSV files"""
    try:
        inventory_df.to_csv(inventory_output, index=False)
        catalogue_df.to_csv(catalogue_output, index=False)
        logger.info(f"Updated inventory saved to: {inventory_output}")
        logger.info(f"Updated catalogue saved to: {catalogue_output}")
    except Exception as e:
        logger.error(f"Error saving files: {e}")
        raise

# Configuration
INVENTORY_FILE = '../../data/processed/library_inventory.csv'
CATALOGUE_FILE = '../../data/processed/library_catalogue.csv'
INVENTORY_OUTPUT = '../../data/processed/library_inventory.csv'
CATALOGUE_OUTPUT = '../../data/processed/library_catalogue.csv'

# Thresholds for fuzzy matching (0-100)
TITLE_THRESHOLD = 85   # Titles need higher confidence
AUTHOR_THRESHOLD = 80  # Authors can be slightly more flexible

try:
    # Load data
    inventory, catalogue = load_data(INVENTORY_FILE, CATALOGUE_FILE)
    
    # Check if required columns exist
    required_cols = ['Title', 'Author']

    for df, name in [(inventory, 'inventory'), (catalogue, 'catalogue')]:
        for col in required_cols:
            if col not in df.columns:
                logger.warning(f"Column '{col}' not found in {name}. Please adjust column names.")
    
    # Integrate data
    inventory_updated, catalogue_updated, stats = integrate_data(
        inventory, 
        catalogue, 
        title_threshold=TITLE_THRESHOLD,
        author_threshold=AUTHOR_THRESHOLD
    )
    
    # Save results
    save_results(inventory_updated, catalogue_updated, INVENTORY_OUTPUT, CATALOGUE_OUTPUT)
    
    logger.info("Integration completed successfully!")
    
except Exception as e:
    logger.error(f"Integration failed: {e}")

2025-12-22 23:25:17,756 - INFO - Inventory loaded: 4176 records
2025-12-22 23:25:17,757 - INFO - Catalogue loaded: 565 records
2025-12-22 23:25:17,772 - ERROR - Integration failed: DataFrame index must be unique for orient='index'.
